In [1]:
from tiled.client import from_profile
c = from_profile("chx") # where is this being used? naming something 'important' "c" is not a good practice...

from pyCHX.chx_packages import *
from tqdm import tqdm

import sys
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/database_processing/") # Need a better way to include these directories
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/AutoRun_functions/")
from database_processing import *
from AutoRun_functions import *

%run /nsls2/data/chx/shared/CHX_Software/packages/environment_management/chx_analysis_setup_test_pyCHX.ipynb
baseDir = _base_path_pass_

/nsls2/conda/envs/2024-2.0-py311-tiled/lib/python3.11/site-packages/databroker/v1.py:72: UserWarning: In databroker 2.x, there are separate notions of 'server' and 'client', and register_handler(...) has no effect on the client. Likely this is being done for you on the server side, so you should not worry about this message unless you encounter trouble loading large array data.
  warnings.warn(


running on: jupyter_hub   environment: standard

setting '_base_path_' as /nsls2/data/chx/legacy/analysis/
setting '_base_path_pass_' as /nsls2/data/chx/proposals/
setting '_mask_path_' as /nsls2/data/chx/shared/CHX_Setup/Detector_masks/

Re-imported pyCHX from /nsls2/data/chx/shared/CHX_Software/packages/pyCHX/
Re-imported chx_compress from /nsls2/data/chx/shared/CHX_Software/packages/pyCHX/chx_compress.py

environment dependent settings and patches:
using "%matplotlib inline" for plotting


/nsls2/conda/envs/2024-2.0-py311-tiled/lib/python3.11/site-packages/databroker/v1.py:72: UserWarning: In databroker 2.x, there are separate notions of 'server' and 'client', and register_handler(...) has no effect on the client. Likely this is being done for you on the server side, so you should not worry about this message unless you encounter trouble loading large array data.
  warnings.warn(


## Setup

In [2]:
direction = 'up' # 'up' or 'down': how to run through list of uids available for processing; up: from bottom to top ('oldest first'); down: from top to bottom ('newest first')
list_length = 'auto' # length N of list with uids to be processed at which this notebook will do processing; 'auto': direction='up' -> N=2; direction='down' -> N=0
track_progress = True  # IF True: track progress via entires in json dictionary [currently only implemented in new XPCS_SAXS_auto processing notebook]

end_of_processing_uid='none' # stops if only this uid is left in compressed_uid_list, 'none': not looking for uid, just for empty list timeout
empty_list_timeout= 3600 * 24 * 6  #[s] stops if compressed_uid_list is empty for x s
clear_analysis_in_progress = True # clear database entry with analysis currently ongoing

start_fuid = 'ccdc16d2-df0d-48ff-8e8c-142091ea3119' # uid from which automated analysis will be started (next uid after start_uid)

In [3]:
assert direction in ['up','down'], "direction must be either 'up' or 'down'"
assert list_length == 'auto' or type(list_length)==int,"list_length must be either 'auto' or an integer"
data_acquisition_collection = import_sample_database()
txt_filename = 'AutoRuns_%sList_%s.txt'%(direction,machine)
if alternate_directory:
    txt_filename = 'Raydata_AutoRuns_%sList_%s.txt'%(direction,machine)

available databases:
Database(MongoClient(host=['mongo3.nsls2.bnl.gov:27017', 'mongo1.nsls2.bnl.gov:27017', 'mongo2.nsls2.bnl.gov:27017'], document_class=dict, tz_aware=False, connect=True), 'database_names')

 available collection in database samples:
Collection(Database(MongoClient(host=['mongo3.nsls2.bnl.gov:27017', 'mongo1.nsls2.bnl.gov:27017', 'mongo2.nsls2.bnl.gov:27017'], document_class=dict, tz_aware=False, connect=True), 'chx-samples'), 'collection_names')


### all uids awaiting processing:

In [4]:
list_database_uids(data_acquisition_collection)

uids to be processed that have not been completed or failed:  []
More than one device. This would have unintented consequences.Currently, only the device contains 'default_dec=eiger'.
More than one device. This would have unintented consequences.Currently, only the device contains 'default_dec=eiger'.
last processed: scan_id: 163340 / uid: 2e74c07c-0891-4b3b-9b63-831a141ac746
last completed: scan_id: 163339 / uid: caec0de1-e7b2-407d-be95-e12f175d5cef
last failed: scan_id: 163340 / uid: 2e74c07c-0891-4b3b-9b63-831a141ac746


### uids after start_uid awaiting processing:

In [5]:
get_masked_analysis_database( start_uid = start_fuid, data_acquisition_collection = data_acquisition_collection)
list_database_uids(data_acquisition_collection,verbose=True)

10684 ccdc16d2-df0d-48ff-8e8c-142091ea3119
uids to be processed that have not been completed or failed:  ['9ed1bdf0-72ee-41ad-8ec9-d0808e862416', '2e74c07c-0891-4b3b-9b63-831a141ac746']
More than one device. This would have unintented consequences.Currently, only the device contains 'default_dec=eiger'.
last processed: scan_id: 163340 / uid: 2e74c07c-0891-4b3b-9b63-831a141ac746
last completed: scan_id: 163339 / uid: caec0de1-e7b2-407d-be95-e12f175d5cef
last failed: scan_id: 163201 / uid: ccdc16d2-df0d-48ff-8e8c-142091ea3119
datasets awaiting processing: 
More than one device. This would have unintented consequences.Currently, only the device contains 'default_dec=eiger'.
[0] scan_id: 163306 / uid: 9ed1bdf0-72ee-41ad-8ec9-d0808e862416     0.005s x 200 fr. Trans:1.0 sample: none 
More than one device. This would have unintented consequences.Currently, only the device contains 'default_dec=eiger'.
[1] scan_id: 163340 / uid: 2e74c07c-0891-4b3b-9b63-831a141ac746     0.005s x 200 fr. Trans:1

### uids currently being processed:

In [6]:
if clear_analysis_in_progress:
    data_acquisition_collection.update_one( {'_id':'general_list'},{'$set':{'analysis_in_progress':  [] }   })

print('Analysis in progress for uids: ',data_acquisition_collection.find_one({'_id':'general_list'})['analysis_in_progress'])

Analysis in progress for uids:  []


# Data Processing from data-acquisition Database 

In [7]:
run_papermill_loop(data_acquisition_collection,direction,list_length,end_of_processing_uid,empty_list_timeout,txt_filename,baseDir,alternate_directory,machine=machine,track_progress=track_progress,verbose=True)

list of uids for analysis is emtpy...going to look again in 5s.


KeyboardInterrupt: 